In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
h2_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H2")
h2_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
macro = pd.read_csv(base_dir / "Dataset Maestro.csv")

# ===== TIPOS =====
macro["price"] = pd.to_numeric(macro["price"], errors="coerce")
macro["rating"] = pd.to_numeric(macro["rating"], errors="coerce")
macro["review_count"] = pd.to_numeric(macro["review_count"], errors="coerce")
macro["subcategory"] = macro["subcategory"].astype("string").str.strip()
macro["brand"] = macro["brand"].astype("string").str.strip()

# ===== LIMPIEZA =====
df = macro.dropna(subset=["subcategory", "brand", "price"]).copy()

# ===== GAMAS DE PRECIO POR SUBCATEGORÍA =====
percentiles = (
    df.groupby("subcategory")["price"]
    .quantile([0.50, 0.90])
    .unstack()
    .rename(columns={0.50: "p50", 0.90: "p90"})
    .reset_index()
)

df = df.merge(percentiles, on="subcategory", how="left")

def asignar_gama(row):
    if pd.isna(row["price"]) or pd.isna(row["p50"]) or pd.isna(row["p90"]):
        return pd.NA
    if row["price"] >= row["p90"]:
        return "alta"
    elif row["price"] >= row["p50"]:
        return "media"
    else:
        return "baja"

df["price_tier"] = df.apply(asignar_gama, axis=1)

# ===== TOP 10% POPULARIDAD POR SUBCATEGORÍA =====
q90_reviews = (
    df.groupby("subcategory")["review_count"]
    .quantile(0.90)
    .rename("q90_review_count")
    .reset_index()
)

df = df.merge(q90_reviews, on="subcategory", how="left")
df["top_10_popularity"] = df["review_count"] >= df["q90_review_count"]

# ===== TABLA GLOBAL H2 =====
h2_global = (
    df.groupby(["subcategory", "price_tier", "brand"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_review_count=("review_count", "mean"),
        avg_rating=("rating", "mean"),
        pct_top_10=("top_10_popularity", "mean"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

h2_global["pct_top_10"] = h2_global["pct_top_10"] * 100

# ===== TABLAS POR GAMA =====
h2_alta = h2_global[h2_global["price_tier"] == "alta"].copy()
h2_media = h2_global[h2_global["price_tier"] == "media"].copy()
h2_baja = h2_global[h2_global["price_tier"] == "baja"].copy()

# ===== TOP MARCAS POR SUBCATEGORÍA =====
top_marcas = (
    df.groupby(["subcategory", "brand"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_review_count=("review_count", "mean"),
        avg_rating=("rating", "mean"),
        pct_top_10=("top_10_popularity", "mean"),
    )
    .reset_index()
)

top_marcas["pct_top_10"] = top_marcas["pct_top_10"] * 100
top_marcas = top_marcas.sort_values(["subcategory", "pct_top_10", "avg_review_count"], ascending=[True, False, False])

# ===== GUARDAR CSVs =====
h2_global.to_csv(h2_dir / "h2_resumen_global.csv", index=False, encoding="utf-8-sig")
h2_alta.to_csv(h2_dir / "h2_gama_alta.csv", index=False, encoding="utf-8-sig")
h2_media.to_csv(h2_dir / "h2_gama_media.csv", index=False, encoding="utf-8-sig")
h2_baja.to_csv(h2_dir / "h2_gama_baja.csv", index=False, encoding="utf-8-sig")
top_marcas.to_csv(h2_dir / "h2_top_marcas.csv", index=False, encoding="utf-8-sig")

print(f"✅ Archivos H2 guardados en: {h2_dir}")
print(h2_global.head())

✅ Archivos H2 guardados en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H2
  subcategory price_tier     brand  n_products  avg_review_count  avg_rating  \
0      Cascos       alta  Logitech           4              39.0         NaN   
1      Cascos       baja     Apple           1              39.0         NaN   
2      Cascos       baja      Cool           5              39.0         NaN   
3      Cascos       baja    HyperX           1              39.0         NaN   
4      Cascos       baja      Krom           2              39.0         NaN   

   pct_top_10  median_price  avg_price  
0       100.0      202490.0   186240.0  
1       100.0        5790.0     5790.0  
2       100.0       22940.0    25780.0  
3       100.0        6890.0     6890.0  
4       100.0       16985.0    16985.0  


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
h2_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H2")
h2_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
macro = pd.read_csv(base_dir / "Dataset Maestro.csv")

# ===== TIPOS =====
macro["price"] = pd.to_numeric(macro["price"], errors="coerce")
macro["rating"] = pd.to_numeric(macro["rating"], errors="coerce")
macro["review_count"] = pd.to_numeric(macro["review_count"], errors="coerce")
macro["subcategory"] = macro["subcategory"].astype("string").str.strip()
macro["brand"] = macro["brand"].astype("string").str.strip()

# ===== LIMPIEZA =====
df = macro.dropna(subset=["subcategory", "brand", "price"]).copy()

# ===== PERCENTILES POR SUBCATEGORÍA =====
percentiles = (
    df.groupby("subcategory")["price"]
    .quantile([0.50, 0.90])
    .unstack()
    .rename(columns={0.50: "p50", 0.90: "p90"})
    .reset_index()
)

df = df.merge(percentiles, on="subcategory", how="left")

def asignar_gama(row):
    if pd.isna(row["price"]) or pd.isna(row["p50"]) or pd.isna(row["p90"]):
        return pd.NA
    if row["price"] >= row["p90"]:
        return "alta"
    elif row["price"] >= row["p50"]:
        return "media"
    else:
        return "baja"

df["price_tier"] = df.apply(asignar_gama, axis=1)

# ===== TOP 10% POPULARIDAD =====
q90_reviews = (
    df.groupby("subcategory")["review_count"]
    .quantile(0.90)
    .rename("q90_review_count")
    .reset_index()
)

df = df.merge(q90_reviews, on="subcategory", how="left")
df["top_10_popularity"] = df["review_count"] >= df["q90_review_count"]

# ===== TABLAS H2 =====
h2_global = (
    df.groupby(["subcategory", "price_tier", "brand"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_review_count=("review_count", "mean"),
        avg_rating=("rating", "mean"),
        pct_top_10=("top_10_popularity", "mean"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

h2_global["pct_top_10"] = h2_global["pct_top_10"] * 100

h2_alta = h2_global[h2_global["price_tier"] == "alta"].copy()
h2_media = h2_global[h2_global["price_tier"] == "media"].copy()
h2_baja = h2_global[h2_global["price_tier"] == "baja"].copy()

top_marcas = (
    df.groupby(["subcategory", "brand"], dropna=False)
    .agg(
        n_products=("product_name", "count"),
        avg_review_count=("review_count", "mean"),
        avg_rating=("rating", "mean"),
        pct_top_10=("top_10_popularity", "mean"),
    )
    .reset_index()
)

top_marcas["pct_top_10"] = top_marcas["pct_top_10"] * 100
top_marcas = top_marcas.sort_values(["subcategory", "pct_top_10", "avg_review_count"], ascending=[True, False, False])

# ===== GUARDAR CSV =====
h2_global.to_csv(h2_dir / "h2_resumen_global.csv", index=False, encoding="utf-8-sig")
h2_alta.to_csv(h2_dir / "h2_gama_alta.csv", index=False, encoding="utf-8-sig")
h2_media.to_csv(h2_dir / "h2_gama_media.csv", index=False, encoding="utf-8-sig")
h2_baja.to_csv(h2_dir / "h2_gama_baja.csv", index=False, encoding="utf-8-sig")
top_marcas.to_csv(h2_dir / "h2_top_marcas.csv", index=False, encoding="utf-8-sig")

# ===== FUNCIÓN PNG =====
def save_table_png(df_table, filename, figsize=None):
    if figsize is None:
        figsize = (14, max(4, 0.35 * len(df_table) + 1.5))
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")
    table = ax.table(cellText=df_table.values, colLabels=df_table.columns, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.25)
    plt.tight_layout()
    fig.savefig(h2_dir / filename, dpi=200, bbox_inches="tight")
    plt.close(fig)

# ===== GUARDAR PNG =====
save_table_png(h2_global, "h2_resumen_global.png")
save_table_png(h2_alta, "h2_gama_alta.png")
save_table_png(h2_media, "h2_gama_media.png")
save_table_png(h2_baja, "h2_gama_baja.png")
save_table_png(top_marcas.head(30), "h2_top_marcas_top30.png", figsize=(16, 12))

print(f"✅ Archivos H2 guardados en: {h2_dir}")
print(h2_global.head())

✅ Archivos H2 guardados en: C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H2
  subcategory price_tier     brand  n_products  avg_review_count  avg_rating  \
0      Cascos       alta  Logitech           4              39.0         NaN   
1      Cascos       baja     Apple           1              39.0         NaN   
2      Cascos       baja      Cool           5              39.0         NaN   
3      Cascos       baja    HyperX           1              39.0         NaN   
4      Cascos       baja      Krom           2              39.0         NaN   

   pct_top_10  median_price  avg_price  
0       100.0      202490.0   186240.0  
1       100.0        5790.0     5790.0  
2       100.0       22940.0    25780.0  
3       100.0        6890.0     6890.0  
4       100.0       16985.0    16985.0  
